# 🏰 Citadel Governance Hub - Backend Onboarding Runner

## Explore how to bring new AI backends into AI Citadel Governance Hub!

Use this Jupyter notebook with Python code snippets to onboard all your AI backends and all associated routing logic.

> **Note:** This notebook assumes you have already set up your Citadel Governance Hub and have models deployed and managed through it. If you haven't done so, please refer to the [Citadel Governance Hub Deployment Guide](../guides/full-deployment-guide.md) or [Citadel Governance Hub Quick Deployment Guide](../guides/quick-deployment-guide.md) before proceeding.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

>Pro tip: If you used Azure Developer CLI ```azd up``` to deploy your Citadel Governance Hub, you can retrieve these values using the command ```azd env get-value LLM_BACKEND_CONFIG ``` to get the backend configuration JSON for the provisioned models.

In [1]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# REQUIRED: Update these values for your environment
# ============================================================================
governance_hub_resource_group = "rg-citadel-hub-dev-1"  ## specify the resource group name where the Governance Hub is located
location = "swedencentral"  ## e.g., "eastus", "westus2", etc.

# ============================================================================
# OPTIONAL: LLM Backend Configuration (pre-configured with sample values from main.bicepparam)
# These values match the AI Foundry backends defined in the template
#
# Required Properties:
# - backendId: Unique identifier (used in APIM backend resource name)
# - backendType: 'ai-foundry' | 'azure-openai' | 'external'
# - endpoint: Base URL of the LLM service
# - authScheme: 'managedIdentity' | 'apiKey' | 'token'
# - supportedModels: Array of model objects with per-model metadata
#
# Model Object Properties:
# - name: Model name (required)
# - sku: SKU name for the deployment (default: 'Standard')
# - capacity: Capacity/TPM quota (default: 100)
# - modelFormat: Model format identifier, e.g., 'OpenAI', 'DeepSeek', 'Microsoft' (default: 'OpenAI')
# - modelVersion: Version of the model (default: '1')
#
# Optional Properties (for load balancing):
# - priority: 1-5, default 1 (lower = higher priority)
# - weight: 1-1000, default 100 (higher = more traffic share)
# ============================================================================
llm_backends_config = [
    {
        "backendId": "aif-bc2yajbbxycfw-0",
        "backendType": "ai-foundry",
        "endpoint": "https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/",  # Replace with your endpoint
        "authScheme": "managedIdentity",
        "supportedModels": [
            { "name": "gpt-4o-mini", "sku": "GlobalStandard", "capacity": 100, "modelFormat": "OpenAI", "modelVersion": "2024-07-18" },
            { "name": "gpt-4o", "sku": "GlobalStandard", "capacity": 100, "modelFormat": "OpenAI", "modelVersion": "2024-11-20" },
            { "name": "DeepSeek-R1", "sku": "GlobalStandard", "capacity": 1, "modelFormat": "DeepSeek", "modelVersion": "1" },
            { "name": "Phi-4", "sku": "GlobalStandard", "capacity": 1, "modelFormat": "Microsoft", "modelVersion": "3" },
            { "name": "text-embedding-3-large", "sku": "GlobalStandard", "capacity": 100, "modelFormat": "OpenAI", "modelVersion": "1" }
        ],
        "priority": 1,
        "weight": 100
    },
    {
        "backendId": "aif-bc2yajbbxycfw-1",
        "backendType": "ai-foundry",
        "endpoint": "https://aif-bc2yajbbxycfw-1.cognitiveservices.azure.com/",  # Replace with your endpoint
        "authScheme": "managedIdentity",
        "supportedModels": [
            { "name": "gpt-5", "sku": "GlobalStandard", "capacity": 100, "modelFormat": "OpenAI", "modelVersion": "2025-08-07" },
            { "name": "DeepSeek-R1", "sku": "GlobalStandard", "capacity": 1, "modelFormat": "DeepSeek", "modelVersion": "1" },
            { "name": "text-embedding-3-large", "sku": "GlobalStandard", "capacity": 100, "modelFormat": "OpenAI", "modelVersion": "1" }
        ],
        "priority": 2,
        "weight": 50
    }
]

# Managed Identity for APIM authentication (will be auto-discovered if not specified)
apim_managed_identity_name = ""  # Leave empty to auto-discover

utils.print_info(f"Initialization completed.")

👉🏽 Initialization completed.


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 16:18:25.431682 :0s]
👉🏽 Current user: admin@MngEnvMCAP805053.onmicrosoft.com
👉🏽 Tenant ID: fcbfc09a-8ad2-4370-929e-1946f1aa1f6f
👉🏽 Subscription ID: ad44bcb2-464c-463f-86b8-a89d43903cbf


<a id='init'></a>
### ⚙️ Initialize APIM Client Tool

👉 Initialize the APIM client to interact with your existing Governance Hub deployment:

In [3]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 16:18:25.641213 :0s]
👉🏽 Current user: admin@MngEnvMCAP805053.onmicrosoft.com
👉🏽 Tenant ID: fcbfc09a-8ad2-4370-929e-1946f1aa1f6f
👉🏽 Subscription ID: ad44bcb2-464c-463f-86b8-a89d43903cbf
⚙️ Running: az resource list -g rg-citadel-hub-dev-1 --resource-type Microsoft.ApiManagement/service 
✅ Listing APIM Resources ⌚ 16:18:27.540618 :1s]
👉🏽 APIM Service Id: /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw
👉🏽 APIM Gateway URL: https://apim-bc2yajbbxycfw.azure-api.net
👉🏽 Retrieved key 0 for subscription: 6989a6ad5c87ed004b070001
👉🏽 Retrieved key 1 for subscription: 6989a6ad5c87ed004b070002
👉🏽 Retrieved key 2 for subscription: master
👉🏽 Retrieved key 3 for subscription: 69933150217d200328e2689f
👉🏽 Retrieved key 4 for subscription: LLM-CRM-SupportAgent-DEV-SUB-01
✅ APIM Client Tool initialized successfully! ⌚ 16:18:31.218822 
👉🏽 APIM Resource 

<a id='2'></a>
### 2️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [4]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

👉🏽 Extracting current APIM backends configuration...
👉🏽 Retrieving APIM backends using Azure SDK...
👉🏽 🔗 Backend: aif-bc2yajbbxycfw-0 -> https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/
👉🏽 🔗 Backend: aif-bc2yajbbxycfw-1 -> https://aif-bc2yajbbxycfw-1.cognitiveservices.azure.com/
👉🏽 🔗 Backend: content-safety-backend -> https://cog-consafety-bc2yajbbxycfw.cognitiveservices.azure.com/
👉🏽 📦 Backend Pool: DeepSeek-R1-backend-pool (2 backends)
👉🏽 🔗 Backend: foundry-embeddings -> https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
👉🏽 🔗 Backend: ms-learn-mcp-server -> https://learn.microsoft.com/api/mcp
👉🏽 📦 Backend Pool: text-embedding-3-large-backend-pool (2 backends)
✅ Found 5 individual backends and 2 backend pools ⌚ 16:18:31.868221 


In [5]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 6 unique supported models
✅ Supported models in APIM policy fragment 'set-backend-pools': ⌚ 16:18:32.857380 
  • DeepSeek-R1
  • Phi-4
  • gpt-4o
  • gpt-4o-mini
  • gpt-5
  • text-embedding-3-large


In [6]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • aif-bc2yajbbxycfw-0
    URL: https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/
    Models: gpt-4o-mini, gpt-4o, DeepSeek-R1, Phi-4, text-embedding-3-large
  • aif-bc2yajbbxycfw-1
    URL: https://aif-bc2yajbbxycfw-1.cognitiveservices.azure.com/
    Models: gpt-5, DeepSeek-R1, text-embedding-3-large
  • content-safety-backend
    URL: https://cog-consafety-bc2yajbbxycfw.cognitiveservices.azure.com/
  • foundry-embeddings
    URL: https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
  • ms-learn-mcp-server
    URL: https://learn.microsoft.com/api/mcp

📦 Backend Pools:
  • DeepSeek-R1-backend-pool
    - /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw/backends/aif-bc2yajbbxycfw-0 (priority

<a id='3'></a>
### 3️⃣ Discover Managed Identity for APIM Authentication

Auto-discover or specify the user-assigned managed identity used by APIM:

In [7]:
# Discover managed identity from APIM using the SDK
utils.print_info("Discovering managed identity configuration...")

# Use the APIMClientTool's get_managed_identity_info method
managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name') or apim_managed_identity_name
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover managed identity. Please specify it manually in the configuration.")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

👉🏽 Discovering managed identity configuration...
✅ Found user-assigned managed identity: id-apim-bc2yajbbxycfw ⌚ 16:18:33.198944 
👉🏽 Client ID: b67c85fc-cef5-4c0a-85fd-fe647d07cf35
✅ Managed Identity Name: id-apim-bc2yajbbxycfw ⌚ 16:18:33.199142 
✅ Managed Identity Resource Group: rg-citadel-hub-dev-1 ⌚ 16:18:33.199148 


<a id='4'></a>
### 4️⃣ Generate LLM Backend Parameter File

Generate a customizable `.bicepparam` file with the full list of LLM backends to be integrated with APIM:

In [8]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    print(f"     Auth: {backend['authScheme']}")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver})")
        else:
            print(f"       - {model_name}")

👉🏽 LLM Backends to be deployed:

  🔗 aif-bc2yajbbxycfw-0
     Type: ai-foundry
     Endpoint: https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/
     Auth: managedIdentity
     Priority: 1, Weight: 100
     Models (5):
       - gpt-4o-mini (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 2024-07-18)
       - gpt-4o (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 2024-11-20)
       - DeepSeek-R1 (SKU: GlobalStandard, Capacity: 1, Format: DeepSeek, Version: 1)
       - Phi-4 (SKU: GlobalStandard, Capacity: 1, Format: Microsoft, Version: 3)
       - text-embedding-3-large (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 1)

  🔗 aif-bc2yajbbxycfw-1
     Type: ai-foundry
     Endpoint: https://aif-bc2yajbbxycfw-1.cognitiveservices.azure.com/
     Auth: managedIdentity
     Priority: 2, Weight: 50
     Models (3):
       - gpt-5 (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 2025-08-07)
       - DeepSeek-R1 (SKU: GlobalStandard, Ca

In [9]:
# Generate the .bicepparam file content
bicep_dir = "../bicep/infra/llm-backend-onboarding"
params_file = os.path.join(bicep_dir, "llm-backends-generated-local.bicepparam")

# Format a single model object for Bicep
def format_model_for_bicep(model):
    """Format a model object for Bicep with per-model metadata."""
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name: '{model}' }}"
    
    # New format: model object with metadata
    parts = [f"name: '{model['name']}'"]
    if 'sku' in model:
        parts.append(f"sku: '{model['sku']}'")
    if 'capacity' in model:
        parts.append(f"capacity: {model['capacity']}")
    if 'modelFormat' in model:
        parts.append(f"modelFormat: '{model['modelFormat']}'")
    if 'modelVersion' in model:
        parts.append(f"modelVersion: '{model['modelVersion']}'")
    
    return "{ " + ", ".join(parts) + " }"

# Format backends array for Bicep (uses per-model metadata)
def format_backend_for_bicep(backend):
    """Format a backend configuration for Bicep with per-model metadata."""
    # Format each model with its individual metadata
    models_formatted = [format_model_for_bicep(m) for m in backend['supportedModels']]
    models_str = "\n      ".join(models_formatted)
    
    return f"""  {{
    backendId: '{backend['backendId']}'
    backendType: '{backend['backendType']}'
    endpoint: '{backend['endpoint']}'
    authScheme: '{backend['authScheme']}'
    supportedModels: [
      {models_str}
    ]
    priority: {backend.get('priority', 1)}
    weight: {backend.get('weight', 100)}
  }}"""

backends_bicep_str = "\n".join([format_backend_for_bicep(b) for b in llm_backends_config])

params_content = f"""using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

// ============================================================================
// APIM Managed Identity Configuration
// ============================================================================
param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

// ============================================================================
// LLM Backend Configuration Array
// Each model in supportedModels has its own metadata (sku, capacity, modelFormat, modelVersion)
// ============================================================================
param llmBackendConfig = [
{backends_bicep_str}
]

// ============================================================================
// Circuit Breaker Configuration
// ============================================================================
param configureCircuitBreaker = true
"""

# Write the parameter file
utils.print_info(f"Generating parameter file: {params_file}")
with open(params_file, 'w') as f:
    f.write(params_content)

utils.print_ok(f"Parameter file generated successfully!")
print("\n" + "="*60)
print("GENERATED PARAMETER FILE CONTENT:")
print("="*60)
print(params_content)

👉🏽 Generating parameter file: ../bicep/infra/llm-backend-onboarding/llm-backends-generated-local.bicepparam
✅ Parameter file generated successfully! ⌚ 16:18:33.211390 

GENERATED PARAMETER FILE CONTENT:
using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: 2026-02-16 16:18:33
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {
  subscriptionId: 'ad44bcb2-464c-463f-86b8-a89d43903cbf'
  resourceGroupName: 'rg-citadel-hub-dev-1'
  name: 'apim-bc2yajbbxycfw'
}

// ============================================================================
// APIM Managed Identity Configuration
// ==========================================================

<a id='5'></a>
### 5️⃣ Deploy LLM Backend Onboarding Bicep

Deploy the LLM backends, backend pools, and policy fragments to APIM:

In [10]:
# Deploy the LLM backend onboarding
deployment_name = f"llm-backend-onboarding-{time.strftime('%Y%m%d%H%M%S')}"
template_file = os.path.join(bicep_dir, "main.bicep")

utils.print_info(f"Starting deployment: {deployment_name}")
utils.print_info(f"Template: {template_file}")
utils.print_info(f"Parameters: {params_file}")

# Run the subscription-level deployment
deployment_cmd = f"az deployment sub create --name {deployment_name} --location {location} --template-file {template_file} --parameters {params_file}"

output = utils.run(
    deployment_cmd,
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

if output.success:
    utils.print_ok("Deployment completed successfully!")
    
    # Display deployment outputs if available
    outputs = output.json_data.get('properties', {}).get('outputs', {}) if output.json_data else {}
    
    if outputs:
        print("\n" + "="*60)
        print("DEPLOYMENT OUTPUTS:")
        print("="*60)
        
        for key, value in outputs.items():
            print(f"  {key}: {value.get('value')}")
    else:
        utils.print_info("No deployment outputs returned.")
else:
    utils.print_error("Deployment failed. Check the error messages above.")

👉🏽 Starting deployment: llm-backend-onboarding-20260216161833
👉🏽 Template: ../bicep/infra/llm-backend-onboarding/main.bicep
👉🏽 Parameters: ../bicep/infra/llm-backend-onboarding/llm-backends-generated-local.bicepparam
⚙️ Running: az deployment sub create --name llm-backend-onboarding-20260216161833 --location swedencentral --template-file ../bicep/infra/llm-backend-onboarding/main.bicep --parameters ../bicep/infra/llm-backend-onboarding/llm-backends-generated-local.bicepparam 
✅ Deployment 'llm-backend-onboarding-20260216161833' succeeded ⌚ 16:19:44.369608 :11s]
✅ Deployment completed successfully! ⌚ 16:19:44.370082 

DEPLOYMENT OUTPUTS:
  apimGatewayUrl: https://apim-bc2yajbbxycfw.azure-api.net
  apimServiceName: apim-bc2yajbbxycfw
  backendIds: ['aif-bc2yajbbxycfw-0', 'aif-bc2yajbbxycfw-1']
  modelToBackendMap: {'Phi-4': 'aif-bc2yajbbxycfw-0', 'gpt-4o': 'aif-bc2yajbbxycfw-0', 'gpt-4o-mini': 'aif-bc2yajbbxycfw-0', 'gpt-5': 'aif-bc2yajbbxycfw-1'}
  modelToPoolMap: {'DeepSeek-R1': 'DeepS

<a id='6'></a>
### 6️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [11]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 16:19:44.652361 :0s]
👉🏽 Current user: admin@MngEnvMCAP805053.onmicrosoft.com
👉🏽 Tenant ID: fcbfc09a-8ad2-4370-929e-1946f1aa1f6f
👉🏽 Subscription ID: ad44bcb2-464c-463f-86b8-a89d43903cbf
👉🏽 APIM Service Id: /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw
👉🏽 APIM Gateway URL: https://apim-bc2yajbbxycfw.azure-api.net
👉🏽 Retrieved key 0 for subscription: 6989a6ad5c87ed004b070001
👉🏽 Retrieved key 1 for subscription: 6989a6ad5c87ed004b070002
👉🏽 Retrieved key 2 for subscription: master
👉🏽 Retrieved key 3 for subscription: 69933150217d200328e2689f
👉🏽 Retrieved key 4 for subscription: LLM-CRM-SupportAgent-DEV-SUB-01
👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 6 unique supported models
✅ Updated supported models in APIM policy fragment 'set-backend-pools': ⌚ 16:19:49.568380 
  • DeepSeek-R1
  • Phi-4
  • gpt-4o
  • gpt-4o-mini
  • 

In [12]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • aif-bc2yajbbxycfw-0
    URL: https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/
    Models: gpt-4o-mini, gpt-4o, DeepSeek-R1, Phi-4, text-embedding-3-large
  • aif-bc2yajbbxycfw-1
    URL: https://aif-bc2yajbbxycfw-1.cognitiveservices.azure.com/
    Models: gpt-5, DeepSeek-R1, text-embedding-3-large
  • content-safety-backend
    URL: https://cog-consafety-bc2yajbbxycfw.cognitiveservices.azure.com/
  • foundry-embeddings
    URL: https://aif-bc2yajbbxycfw-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
  • ms-learn-mcp-server
    URL: https://learn.microsoft.com/api/mcp

📦 Backend Pools:
  • DeepSeek-R1-backend-pool
    - /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw/backends/aif-bc2yajbbxycfw-0 (priority

<a id='7'></a>
### 7️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [13]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

✅ Policy fragment 'get-available-models' exists! ⌚ 16:19:49.718712 
👉🏽 This fragment returns available model deployments in a format similar to Azure Cognitive Services API.
👉🏽 Description: Returns a JSON response listing all available model deployments with their capabilities


<a id='test-foundry'></a>
### 🧪 Test GET /deployments (Microsoft Foundry Integration)

Test the `GET /deployments` endpoint which leverages the `get-available-models` policy fragment. This endpoint is used by Microsoft Foundry to discover available model deployments:

In [14]:
# Test GET /deployments endpoint (Microsoft Foundry integration)
# This endpoint uses the get-available-models policy fragment to return available model deployments
# Testing both Universal LLM API and Azure OpenAI API endpoints

def test_get_deployments(base_endpoint, api_name, api_key):
    """Test GET /deployments for a given API endpoint."""
    deployments_url = f"{base_endpoint}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nTesting {api_name} GET /deployments: {deployments_url}")
    
    try:
        response = requests.get(
            deployments_url,
            headers={"api-key": api_key},
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = response.json()
            deployments = data.get("value", [])
            
            utils.print_ok(f"{api_name} GET /deployments returned {len(deployments)} model deployment(s)")
            
            print("\n" + "="*60)
            print(f"AVAILABLE MODEL DEPLOYMENTS ({api_name})")
            print("="*60)
            
            for deployment in deployments:
                print(f"\n📦 Deployment: {deployment.get('name', 'N/A')}")
                print(f"   ID: {deployment.get('id', 'N/A')}")
                print(f"   Type: {deployment.get('type', 'N/A')}")
                
                sku = deployment.get('sku', {})
                if sku:
                    print(f"   SKU: {sku.get('name', 'N/A')} (Capacity: {sku.get('capacity', 'N/A')})")
                
                props = deployment.get('properties', {})
                if props:
                    model = props.get('model', {})
                    if model:
                        print(f"   Model: {model.get('name', 'N/A')} (Format: {model.get('format', 'N/A')}, Version: {model.get('version', 'N/A')})")
                    
                    capabilities = props.get('capabilities', {})
                    if capabilities:
                        caps_list = [k for k, v in capabilities.items() if v == 'true' or v == True]
                        print(f"   Capabilities: {', '.join(caps_list) if caps_list else 'N/A'}")
                    
                    print(f"   Status: {props.get('provisioningState', 'N/A')}")
            
            print("\n" + "="*60)
            utils.print_ok(f"{api_name} GET /deployments test completed successfully!")
            return True
        else:
            utils.print_error(f"{api_name} GET /deployments failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        utils.print_error(f"{api_name} GET /deployments test failed: {str(e)}")
        return False

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[-2].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[-2].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

if api_key:
    # Test 1: Universal LLM API - GET /models/deployments
    apimClientTool.discover_api("models")
    azure_endpoint_models = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Universal LLM API Base Endpoint: {azure_endpoint_models}models")
    test_get_deployments(f"{azure_endpoint_models}models/", "Universal LLM API", api_key)
    
    # Test 2: Azure OpenAI API - GET /openai/deployments
    try:
        apimClientTool.discover_api("openai")
        azure_endpoint_openai = str(apimClientTool.azure_endpoint)
        utils.print_info(f"\nAzure OpenAI API Base Endpoint: {azure_endpoint_openai}openai")
        test_get_deployments(f"{azure_endpoint_openai}openai/", "Azure OpenAI API", api_key)
    except Exception as e:
        utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
else:
    utils.print_warning("Cannot test GET /deployments - missing API key")

✅ Using subscription: 69933150217d200328e2689f ⌚ 16:19:49.739290 
👉🏽 Found API with id /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-bc2yajbbxycfw.azure-api.net/
👉🏽 Universal LLM API Base Endpoint: https://apim-bc2yajbbxycfw.azure-api.net/models
👉🏽 
Testing Universal LLM API GET /deployments: https://apim-bc2yajbbxycfw.azure-api.net/models/deployments?api-version=2024-05-01-preview
Response status: 200 - OK
✅ Universal LLM API GET /deployments returned 2 model deployment(s) ⌚ 16:19:51.028281 

AVAILABLE MODEL DEPLOYMENTS (Universal LLM API)

📦 Deployment: gpt-4o
   ID: aif-bc2yajbbxycfw-0
   Type: ai-foundry
   SKU: GlobalStandard (Capacity: 100)
   Model: gpt-4o (Format: OpenAI, Version: 2024-11-20)
   Capabilities: chatCompletion
   Status: Succeeded

📦 Deployment: DeepSeek-R1
   ID: aif-bc2yajbbxycfw-0


---
## 🧪 Test Deployed Models

The following sections test the deployed models through both the Universal LLM API and Azure OpenAI API endpoints.

<a id='test-universal'></a>
### 🧪 Test via Universal LLM API (models/chat/completions)

Test the deployed models using the Universal LLM API which routes based on the `model` field in the request body:

In [15]:
# Discover the Universal LLM API endpoint
apimClientTool.discover_api("models")
azure_endpoint_models = str(apimClientTool.azure_endpoint)
chat_completions_url_models = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Universal LLM API Endpoint: {chat_completions_url_models}")

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[-1].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[-2].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

👉🏽 Found API with id /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-bc2yajbbxycfw.azure-api.net/
👉🏽 Universal LLM API Endpoint: https://apim-bc2yajbbxycfw.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
✅ Using subscription: 69933150217d200328e2689f ⌚ 16:19:51.842527 


In [16]:
# Test each supported model via Universal LLM API
if api_key and updated_supported_models:
    utils.print_info(f"\nTesting {len(updated_supported_models)} models via Universal LLM API...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
    
    for model_name in updated_supported_models[:3]:  # Test first 3 models
        utils.print_info(f"Testing model: {model_name}")
        
        payload = {
            "model": model_name,
            "messages": test_messages
        }
        
        try:
            response = requests.post(
                chat_completions_url_models,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run tests - missing API key or supported models")

👉🏽 
Testing 6 models via Universal LLM API...

👉🏽 Testing model: DeepSeek-R1
Response status: 200 - OK
  💬 Response: <think>
Okay, the user asked, "What is 2+2? Answer in one word." Let me think. The question is straightforward. The answer is obviously 4. But wait, maybe they want to check if I can follow instructions. They specified one word. "4" is a single word, right? Numbers can be written as words, but "4" is a numeral. Hmm, does the user consider numerals as words here? Well, in many contexts, when people say "one word," they might accept numerals. Let me check. If I write "four," that's a word. But "4" is a numeral. Which one is better? The user might prefer the numeral since it's shorter and more direct. Also, the question is simple math, so the answer is standard. I should go with "4" to be concise and meet the one-word requirement. No need for extra explanation. Just the answer. Alright, I'll respond with "4."
</think>

4
  📍 Backend Region: Sweden Central
✅ Model 'DeepSeek-

<a id='test-openai'></a>
### 🧪 Test via Azure OpenAI API (openai/deployments/{model}/chat/completions)

Test the deployed models using the Azure OpenAI compatible API which uses the deployment name in the URL path:

In [17]:
# Discover the Azure OpenAI API endpoint
try:
    apimClientTool.discover_api("openai")
    azure_endpoint_openai = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Azure OpenAI API Base Endpoint: {azure_endpoint_openai}")
except Exception as e:
    utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
    azure_endpoint_openai = None

👉🏽 Found API with id /subscriptions/ad44bcb2-464c-463f-86b8-a89d43903cbf/resourceGroups/rg-citadel-hub-dev-1/providers/Microsoft.ApiManagement/service/apim-bc2yajbbxycfw/apis/azure-openai-api and path openai
👉🏽 Azure Endpoint with APIM https://apim-bc2yajbbxycfw.azure-api.net/
👉🏽 Azure OpenAI API Base Endpoint: https://apim-bc2yajbbxycfw.azure-api.net/


In [18]:
# Test models via Azure OpenAI API format
if api_key and azure_endpoint_openai and updated_supported_models:
    utils.print_info(f"\nTesting models via Azure OpenAI API format...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
    
    for model_name in updated_supported_models[:3]:  # Test first 3 models
        utils.print_info(f"Testing model: {model_name}")
        
        # Azure OpenAI format uses deployment name in URL path
        chat_completions_url_openai = f"{azure_endpoint_openai}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
        
        payload = {
            "messages": test_messages  # No model field needed - it's in the URL
        }
        
        try:
            response = requests.post(
                chat_completions_url_openai,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run Azure OpenAI API tests - missing API key, endpoint, or supported models")

👉🏽 
Testing models via Azure OpenAI API format...

👉🏽 Testing model: DeepSeek-R1
Response status: 429 - Too Many Requests
❌ Model 'DeepSeek-R1' - FAILED: {"error":{"code":"RateLimitReached","message": "Your requests to DeepSeek-R1 for DeepSeek-R1 in Sweden Central have exceeded the call rate limit for your current AIServices S0 pricing tier. This request was for ChatCompletions_Create under Azure OpenAI API version 2024-05-01-preview. To increase your default rate limit, visit: https://aka.ms/oai/quotaincrease."}}
 ⌚ 16:19:56.387257 
👉🏽 Testing model: Phi-4
Response status: 401 - Unauthorized model access
❌ Model 'Phi-4' - FAILED: {
  "error": {
    "message": "Access to model 'Phi-4' is not allowed for this product.",
    "type": "access_error",
    "code": "unauthorized_model_access",
    "allowed_models": "gpt-4o,deepseek-r1"
  }
}
 ⌚ 16:19:56.594995 
👉🏽 Testing model: gpt-4o
Response status: 503 - ServiceUnavailable
❌ Model 'gpt-4o' - FAILED: 
 ⌚ 16:19:56.901600 


<a id='test-sdk'></a>
### 🧪 Test using Azure OpenAI Python SDK

Test using the official Azure OpenAI Python SDK:

In [19]:
from openai import AzureOpenAI

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing with Azure OpenAI SDK using model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version
        )
        
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from Azure OpenAI SDK!'"}
            ]
        )
        
        utils.print_ok("SDK Test Successful!")
        print(f"💬 Response: {response.choices[0].message.content}")
        print(f"📊 Usage: {response.usage.total_tokens} tokens")
        
    except Exception as e:
        utils.print_error(f"SDK Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run SDK test - missing prerequisites")

👉🏽 Testing with Azure OpenAI SDK using model: DeepSeek-R1
✅ SDK Test Successful! ⌚ 16:20:02.256837 
💬 Response: <think>
Okay, the user wants me to say "Hello from Azure OpenAI SDK!" Let me make sure I get that right.

First, I need to check if there's any specific formatting or additional information required. The original instruction is straightforward, just to say that exact sentence. 

Wait, maybe they want it in a code block or some specific syntax? The example response uses a code block with "Hello from Azure OpenAI SDK!" inside. But the user didn't specify that. Hmm.

Alternatively, perhaps they just want the plain text response. Since the example uses a code block, maybe I should follow that format. But the user's instruction doesn't mention code formatting. Let me think.

The user's message is "Say 'Hello from Azure OpenAI SDK!'" and the example response is in a code block. Maybe they expect the same way. But to be safe, I'll provide both a plain text and a code block version, 

<a id='test-streaming'></a>
### 🧪 Test Streaming Response

Test streaming responses using the Azure OpenAI SDK:

In [20]:
from openai import AzureOpenAI

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing streaming with model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version
        )
        
        start_time = time.time()
        
        response = client.chat.completions.with_raw_response.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Count from 1 to 10 with commas between each number."}
            ],
            stream=True
        )
        
        print(f"📡 x-ms-region: {response.headers.get('x-ms-region', 'unknown')}")
        print(f"📡 x-ms-stream: {response.headers.get('x-ms-stream', 'N/A')}")
        print("\n💬 Streaming response:")
        
        completion = response.parse()
        collected_content = []
        
        for chunk in completion:
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                collected_content.append(content)
                print(content, end='', flush=True)
        
        elapsed = time.time() - start_time
        print(f"\n\n✅ Stream completed in {elapsed:.2f} seconds")
        print(f"📝 Full response: {''.join(collected_content)}")
        
    except Exception as e:
        utils.print_error(f"Streaming Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run streaming test - missing prerequisites")

👉🏽 Testing streaming with model: DeepSeek-R1
📡 x-ms-region: Sweden Central
📡 x-ms-stream: N/A

💬 Streaming response:
<think>
Okay, the user wants me to count from 1 to 10 with commas between each number. Let me start by recalling the numbers. 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. Wait, I need to make sure each number is separated by a comma. Let me check each number step by step.

First, 1. Then a comma. Next is 2, comma, 3, comma, and so on. Let me write them out: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. Hmm, that look right? Each number is followed by a comma except the last one. Wait, the user said "with commas between each number," so maybe the last number shouldn't have a comma after it. Let me confirm. If I list numbers separated by commas, the standard format is to have commas between them, so 1, 2, 3, ..., 10. So the commas are between the numbers, meaning after each number except the last. So the correct sequence would be 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. That's ten numbers with commas in between

---
## 📊 Summary

This notebook completed the following tasks:

1. ✅ **Extracted** current APIM backend-pools configurations
2. ✅ **Generated** a customizable LLM backend parameter file (`.bicepparam`)
   - Includes per-model metadata fields: `sku`, `capacity`, `modelFormat`, `modelVersion`
3. ✅ **Deployed** the LLM onboarding Bicep templates
4. ✅ **Verified** deployment including the new `get-available-models` policy fragment
5. ✅ **Tested** the deployed models through:
   - **GET /deployments** (Microsoft Foundry integration - uses `get-available-models` policy fragment)
   - Universal LLM API (`/models/chat/completions`)
   - Azure OpenAI API (`/openai/deployments/{model}/chat/completions`)
   - Azure OpenAI Python SDK
   - Streaming responses

### Next Steps

- Modify the `llm_backends_config` in the initialization cell to add more backends
- Include per-model metadata (`sku`, `capacity`, `modelFormat`, `modelVersion`) for accurate model info
- Re-run the deployment cells to update the APIM configuration
- Use the generated parameter file as a template for CI/CD pipelines
- Use the `get-available-models` policy fragment to expose available models to Microsoft Foundry and other clients